# AI Modified Workflow

For this workflow we use the following prompt to modify the `scale_model_size.py` notebook:

Produce an SST weak scaling workflow. Define a global parameter that I can use to set a fixed number of components per node.

---

It's interesting that this prompt updated the name of the directory to conduct the experiment in (i.e. `basedir`) and others didn't.  In this case I like that it did.  Overall, the changes made to the global parameter cell look good.

When I wrote a "weak scaling with PHOLD" notebook, the approach I took to calculating the model size was to compute the width as the square root of the "desired number of components per node" and then increased the height to be the product of that and the number of nodes.  This makes it so each node stores an equally sizeed square-shaped region in some larger rectangular board where all these square shaped regions stack ontop of each other.  For this benchmark, performance is impacted quite a bit by how many components have "cross-partition" (e.g. cross node) links, so for a weak scaling run its important to arrange things so that this value doesn't change dramatically from run to run.

The approach copilot took was quite different.  Instead copilot first compute a global number of desired components and then finds a "best fit" rectangle who's side lengths perfectly factor it.  On the one hand my approach approximates the total desired number of components and on the other Copilot's approach will lead to some odd shaped rectangles.  In the worse case, if your total desired number of components is prime you'll get `width=1` and `height=total_components`.

In practice I've found it to be "close enough" to be practical. With the algorithm Copilot does, it would also take O(n) time to calculate the model size as keeps decreasing the candidate width to find a perfect fit (which it will always find if it gets down to `width=1`).  I'm unsure if this is an issue in practice, but it is something to note.

Copilot also follows up on a poor design choice I made with my weak scaling notebook. Which was I've built things to implicity assume that log files are named `size_<something>`.  For the single node workflows that something is number of components and for weak scaling workflow its number of nodes. But I think I'd prefer weak scaling workflows to name the file something like `nodes_<numNodes>` instead.  Similiarly, in both workflows I just call this "size" in the resulting CSV file and it would be better to be more specific.

Another nitpick, for the results, I think I'd want to update the graph to have the yaxis start at 0.  This could be updated very quickly with a followup prompt though.

---
---
---

# Configuration

## Import workflows module

In [ ]:
from utils.workflows import *

## Global params

Users can modify these top-level parameters to alter the behavior of this workflow.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------------
# !!! DO NOT MODIFY THE CODE BELOW   !!!
# !!!  (Modify in the next section)  !!!
# ---------------------------------------------------------------------------------------------------------------------

# So that can you can maintain the defaults, we suggest you don't directly edit
# the parameters inline here but rather overwite values at the bottom of this
# cell.

# We'll store our containers and benchmark results under the specified directory
# (it will be created if it doesn't already exist).
import os
if 'user_customExperimentsDir' in globals():
    baseDir = f'{user_customExperimentsDir}/weak_scaling'
else:
    baseDir=f'{os.getenv("HOME")}/workflows/weak_scaling'

# Run using an SST in the specified container. To find containers to use see the
# container factory at https://github.com/hpc-ai-adv-dev/sst-container-factory
# Prebuild containers are available at https://github.com/orgs/hpc-ai-adv-dev/packages
container_url  = 'ghcr.io/hpc-ai-adv-dev/sst-core:master-latest'
container_name = None # DO NOT MODIFY THIS LINE: Variable will be assigned after we download the container
                      # We include it here to document what global variables are available throughout the
                      # notebook.

# The benchmark will be cloned from the specified repository. We assume the
# benchmark itself is in the 'benchmarkPath' directory within the repos.  We
# assume building the benchmark is a matter of running 'make' in that directoy.
benchmarkRepos='https://github.com/hpc-ai-adv-dev/sst-benchmarks.git'
benchmarkPath='phold'

# Weak scaling: hold components per node fixed and increase the number of nodes.
components_per_node = 1_000_000
num_nodes_per_trial = [1, 2, 4, 8, 16]

# This command will be run prior to launching a job. The command will be run
# from within the benchmark directory and execution occurs within the worklaunch
# loop so it may be parameterized by the trial parameters if needed.
prestart_cmd_template = ''

# Indicates what arguments should be passed to sst and the benchmark each run 
# Note: {width} and {height} will be replaced with the appropriate values for
# each run, based on the number of nodes and total component count.
sst_args_template   = '--print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py'
bmark_args_template = '--width {width} --height {height}'

# Additional arguments to pass when launching jobs with srun. For example the
# partition name or --qos=high for higher priority in the queue.
additional_srun_args = ''

# Several of the setup steps will avoid rerunning if they have previously been run. Append to this
# list to indicate when you want to force a step to be reproduced.
#
# VALID VALUES ARE:
#   'ALL'     
#   'DOWNLOAD_CONTAINERS' 
#   'DOWNLOAD_BENCHMARKS' 
#   'BUILD_BENCHMARKS'      Note: we always rerun make, if this is set we will also run 'make clean' before rebuilding
force = []

# ---------------------------------------------------------------------------------------------------------------------
# Overwite parameters below this line to customize the workflow: 
# ---------------------------------------------------------------------------------------------------------------------


## Environment

In [ ]:
set_workflow_log(f'{baseDir}/workflow.log')
run_cmd(f'e4s-cl profile edit --add-files {baseDir}')

## Download containers

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_CONTAINERS' in force

container_name = download_custom_container(container_url, force=_force)

## Download benchmarks

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_BENCHMARKS' in force

if not os.path.exists(f'benchmarks') or _force:
    run_cmd(f"git clone {benchmarkRepos} benchmarks")
    run_cmd(f"e4s-cl profile edit --add-files {baseDir}/benchmarks/{benchmarkPath}")
else:
    print(f"Benchmarks from {benchmarkRepos} have already been downloaded, skipping download.")

## Build benchmarks 

In [ ]:
_force = 'ALL' in force or 'BUILD_BENCHMARKS' in force

cd(f"{baseDir}/benchmarks/{benchmarkPath}")
run_cmd('touch sstsimulator.conf')
_cmd = 'make' if not _force else 'make clean; make'
run_in_container(_cmd,
    f'{baseDir}/{container_name}',
    additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')
cd(baseDir)

# Run

## Start jobs

In [ ]:
import math, shutil, os

runDisplay = SafeDisplay(display_handle = display('', display_id="run_disp"))

# Setup directory to store results in
run_dir = f'{baseDir}/runs/'
if os.path.exists(run_dir):
    shutil.rmtree(run_dir)
os.makedirs(run_dir, exist_ok=True)

cd(f"{baseDir}/benchmarks/{benchmarkPath}")

# Deploy jobs
for num_nodes in num_nodes_per_trial:
    total_components = num_nodes * components_per_node

    # Build an exact width/height factorization for the requested component count.
    width = int(math.sqrt(total_components))
    while total_components % width != 0:
        width -= 1
    height = total_components // width

    if prestart_cmd_template is not None and prestart_cmd_template != '':
        run_cmd(prestart_cmd_template.format(
            nodes=num_nodes,
            components_per_node=components_per_node,
            total_components=total_components,
            width=width,
            height=height,
            size=total_components))

    full_sst_args_template = f'{sst_args_template} -- {bmark_args_template}'
    sst_args = full_sst_args_template.format(
        nodes=num_nodes,
        components_per_node=components_per_node,
        total_components=total_components,
        width=width,
        height=height,
        size=total_components)

    launch_and_log_sst(
        image        = f'{baseDir}/{container_name}',
        srun_args    = f'-N {num_nodes} --job-name={benchmarkPath.lower()}_nodes{num_nodes}_size{total_components} {additional_srun_args}',
        sst_args     = sst_args,
        log_file     = f'{run_dir}/size_{num_nodes}',
        config_path  = f'{baseDir}/benchmarks/{benchmarkPath}/sstsimulator.conf',
        safe_display = runDisplay)

cd(f"{baseDir}")

## Watch squeue

In [ ]:
watch_queue_widget()

## Inspect results

In [ ]:
inspect_logs(f'{baseDir}/runs')

# Preprocess

In [ ]:
import os, glob

fullpath = f"{baseDir}/runs"
print(f'\n===== running extract under {fullpath} =====')
cd(fullpath)

data = extract_sst_output_in_files(sorted(glob.glob("size_*")))
csv_lines = convert_to_csv(data)
csv_name = f"{baseDir}/runs/results.csv"

with open(csv_name, 'w') as f:
    f.write('\n'.join(csv_lines))

if os.path.exists(csv_name):
    with open(csv_name, 'r') as f:
        print(f'\n===== {csv_name} =====')
        print(f.read())
else:
    print(f'\n===== {csv_name} (not created) =====')

# Plot

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    df = pd.read_csv(f"{baseDir}/runs/results.csv")
except FileNotFoundError as e:
    print(f'ERROR: File not found - {e.filename}')
    raise StopExecution()

fig = plt.figure()
ax = fig.add_subplot(111)

plot_value='total_duration'
ylabel = 'Total duration (secs)'

ax.scatter(x=df["Size"], y=df[plot_value], c='b', marker="s", label=f'SST 15.1.0')
ax.legend().remove()
plt.title(f'SST {benchmarkPath} weak scaling ({components_per_node} comps/node, {plot_value})')
plt.xlabel('Number of nodes')
plt.ylabel(ylabel)
plt.show()